AI usage: Gemini helped with parallelizing download. I checked all of the modified code to the best of my ability.


In [ ]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.dummy import DummyClassifier

**Download Dataset**

In [ ]:
import os
import requests
import requests
from tqdm.notebook import tqdm

dataset_url = "https://api.biodiversitydata.nl/v2/specimen/dwca/getDataSet/wildlife-sounds-birds"
write_path = "data/wildlife-sounds-birds-20260928.dwca.zip"

try:
    with requests.get(dataset_url, timeout=10, stream=True) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))

        with open(write_path, "wb") as handler:
            with tqdm(
                total=total or None,
                unit="B",
                unit_scale=True,
                desc="Downloading",
            ) as bar:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handler.write(chunk)
                        bar.update(len(chunk))

except requests.RequestException as e:
    print(f"Failed to download {dataset_url}: {e}")

**Extract Data**

In [ ]:
zip_path = write_path

In [ ]:
# import packages first
#!pip install python-dwca-reader

from dwca.read import DwCAReader
import pandas as pd

Create DataFrame (with merged core and extension):

In [ ]:
pd.set_option('display.max_colwidth', None) # avoid truncating the urls

with DwCAReader(zip_path) as dwca:
  # get the 'core'
  df_core = dwca.pd_read(dwca.core_file_location)

  # get the extensions of the core
  ext_file = dwca.extension_files[0] # get name of ext file
  df_ext = dwca.pd_read(str(ext_file))

  # strip out important info and change 'coreid' to 'id'
  datatype = 'Sound'
  df_ext_urls = df_ext[df_ext['type'] == datatype].drop_duplicates(subset=['coreid']) # drop non-audio files and any extra entries
  df_ext_urls = df_ext_urls[['coreid', 'accessURI', 'type', 'rights']]
  df_ext_urls.rename(columns={'coreid':'id'}, inplace = True)

  # merge the core and its extensions on 'id'
  df_merged = pd.merge(df_core[['id', 'scientificName', 'vernacularName']], df_ext_urls, on='id')

df_merged

Code for checking individual counts of bird samples (to ensure there are at least 200):

(not necessary to run)

In [ ]:
count = df_merged['vernacularName'].value_counts().get("Thrush-like Antpitta", 0)
count

In [ ]:
# df_merged['vernacularName'].value_counts()[40:45]

New `DataFrame` with 5 birds and 200 samples for each:

In [ ]:
# selected_birds = ['Common Blackbird', 'Common Chiffchaff', 'Great Tit', 'Common Chaffinch', 'European Robin']
selected_birds = ["Azara's Spinetail", 'Mourning Dove', 'Yellowhammer', 'Eurasian Nuthatch', 'European Robin']

df_sampled = df_merged[df_merged['vernacularName'].isin(selected_birds)].groupby('vernacularName',group_keys=False).apply(lambda x: x.sample(200))
df_sampled

**Download the Data**

In [ ]:
# adapted for parallelization with Gemini

import os
import requests
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

top_dir = 'bird_recordings/'

# create directory structure
for name in selected_birds:
  if not os.path.exists(top_dir+name):
    os.makedirs(top_dir+name)

# Convert sampled dataframe to list of dicts/tuples for easy mapping
download_tasks = []
for counter, bird in enumerate(df_sampled.itertuples(index=False)):
  sub_dir = bird.vernacularName + '/'
  filename = f"{counter}.mp3"
  filepath = os.path.join(top_dir, sub_dir, filename)
  download_tasks.append((bird.accessURI, filepath))

def download_image(task):
  url, filepath = task
  try:
    img_data = requests.get(url, timeout=10).content
    with open(filepath, 'wb') as handler:
      handler.write(img_data)
    return True
  except Exception as e:
    # Return the error or False so we can track issues if necessary
    return (url, str(e))

# download the files in parallel using ThreadPoolExecutor
num_workers = 16  # Adjust based on system resources and server limitations
results = []

with ThreadPoolExecutor(max_workers=num_workers) as executor:
  # Submit all download tasks
  futures = {executor.submit(download_image, task): task for task in download_tasks}

  # Process futures as they complete with a progress bar
  for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading bird spectrograms in parallel"):
    res = future.result()
    if res != True:
      print(f"Failed to download {res[0]}: {res[1]}")

In [ ]:
#!pip install librosa
#Making changes from previous download base
#I did it together with chatgpt because I couldn't get results without large amounts of download errors
import os
import time
import random
import requests
import librosa

from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


#check if an audio file exists and if can be read
def check_audio(filepath):
    try:
        #Check if the file exists
        if not os.path.isfile(filepath):
            return False

        #Check if the file is empty
        if os.path.getsize(filepath) == 0:
            return False

        #Load the complete audio to check for errors
        y, sr = librosa.load(filepath, sr=None)

        #Make sure the audio contains samples
        return len(y) > 0 and sr > 0

    except Exception:
        return False


#Function for Download audio file
def download_audio(task):
    url, filepath = task
    max_attempts = 3
    temp_path = filepath + ".part"

    #Skip files that were already downloaded and validated
    if check_audio(filepath):
        return ("skipped", filepath)

    #Try downloading the file up to three times to maximize downloads
    for attempt in range(max_attempts):
        try:
            #Request the audio file from the server
            response = requests.get(
                url,
                timeout=(5, 20)
            )

            #Check for HTTP errors such as 503
            response.raise_for_status()

            #Check if the downloaded file is empty
            if not response.content:
                return ("failed", filepath, "Empty download")

            #save the file temporarily before replacing anything
            with open(temp_path, "wb") as file:
                file.write(response.content)

            #heck if Librosa can read the downloaded file
            if not check_audio(temp_path):
                return ("failed", filepath, "Invalid audio file")

            #Keep the file only after it passes validation
            os.replace(temp_path, filepath)

            return ("downloaded", filepath)

        except requests.exceptions.RequestException as e:

            #Get the HTTP error code if available
            status = (
                e.response.status_code
                if isinstance(e, requests.exceptions.HTTPError)
                and e.response is not None
                else None
            )

            #Retry temporary server or connection errors
            retryable = (
                status in [429, 500, 502, 503, 504]
                or isinstance(
                    e,
                    (
                        requests.exceptions.Timeout,
                        requests.exceptions.ConnectionError
                    )
                )
            )

            #Stop if the error cannot be retried
            if not retryable or attempt == max_attempts - 1:
                return ("failed", filepath, str(e))

            #wait longer before trying again to unsure proper download
            wait_time = (2 ** attempt) + random.uniform(0, 1)
            time.sleep(wait_time)

        except Exception as e:
            return ("failed", filepath, str(e))

        finally:
            #remove any temporary file left after the attempt
            if os.path.exists(temp_path):
                os.remove(temp_path)

    return ("failed", filepath, "Download failed")


#download audio files in parallel, two workers in parallel
num_workers = 2
results = []

with ThreadPoolExecutor(max_workers=num_workers) as executor:

    #Start all download tasks
    futures = {
        executor.submit(download_audio, task): task
        for task in download_tasks
    }

    #process each task as it finishes
    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Downloading bird audio files"
    ):
        result = future.result()
        results.append(result)

        #Show only failed downloads
        if result[0] == "failed":
            print(f"Failed: {result[1]} - {result[2]}")


#Counter for the download results
downloaded = sum(r[0] == "downloaded" for r in results)
skipped = sum(r[0] == "skipped" for r in results)
failed = sum(r[0] == "failed" for r in results)

#show the final summary of the downloads
print("\nDownload Summary")
print(f"New downloads: {downloaded}")
print(f"Existing files reused: {skipped}")
print(f"Failed downloads: {failed}")
print(f"Total files checked: {len(results)}")


3028 bytes is an empty mp3:

In [ ]:
!ls -l /content/bird_recordings/Eurasian\ Nuthatch


Checking the audio server

In [ ]:
import requests

#Select one audio URL from our dataset
test_url = df_sampled.iloc[0]["accessURI"]

try:
    #send only one request to check the server to see if the problems were the urls
    response = requests.get(
        test_url,
        timeout=30,
        stream=True
    )

    print("HTTP status:", response.status_code)
    print("Content type:", response.headers.get("Content-Type"))
    print("Retry after:", response.headers.get("Retry-After"))

    response.close()

except requests.exceptions.RequestException as e:
    print("Connection error", e)

    #HTTP 200 OK successful response status code indicates that a request has succeeded
    #all good no need further changes within the url

Checking the download tasks 

In [ ]:
#Check how many recordings we want to download
print(f"Total download tasks: {len(download_tasks)}")

#get the URLs from our download tasks
urls = [url for url, filepath in download_tasks]

#Check if any URLs are repeated to avoid conflict
print(f"Unique URLs: {len(set(urls))}")
print(f"Repeated URLs: {len(urls) - len(set(urls))}")


# 200 recordings per bird species, 5 species, 1000 recordings in total (downloads)

Manifest


In [ ]:
import pandas as pd
#Will impliment a manifest help to se our downloads
# Create a table that connects each audio URL to its saved file path
#helping us identify the original recording if a download fails
#also preventing us from mixing recordings when we rerun the notebook
download_manifest = pd.DataFrame(
    download_tasks,
    columns=["url", "filepath"]
)

#Save the table so we can recover failed downloads later
manifest_path = os.path.join(top_dir, "bird_download_manifest.csv")

#Check if we already have a manifest from a previous download
if os.path.exists(manifest_path):
    old_manifest = pd.read_csv(manifest_path)

    #Stop if the selected recordings or file paths have changed unwanted changes
    #preveents us from reusing files that belong to different URLs
    if not old_manifest.equals(download_manifest):
        raise ValueError(
            "the download selection has changed "
            "use a new folder to avoid mixing recordings"
        )
else:
    download_manifest.to_csv(manifest_path, index=False)

print("download manifest ready")

Visualize the names of the birds (5)

In [ ]:
# !pip install librosa
import librosa
import librosa.display
import IPython.display as ipd
import os
for file in os.listdir('bird_recordings/'):
  if file not in ['.ipynb_checkpoints', 'bird_download_manifest.csv']:
    print(file)

Preparing the audio dataset

In [ ]:
import os
import pandas as pd

#Create a list of recordings that were downloaded successfully
audio_data = []

for url, filepath in download_tasks:

    #O0nly include files that exist
    if check_audio(filepath):

        #Get the bird species from the folder name
        species = os.path.basename(os.path.dirname(filepath))

        audio_data.append({
            "filepath": filepath,
            "species": species
        })

#convert the list into our DataFrame
audio_df = pd.DataFrame(audio_data)

#Check how many recordings we have for each species
print("total valid recordings:", len(audio_df))
print("\nrrecordings per species:")
print(audio_df["species"].value_counts())

Splitting dataset into train, validation, and test sets

In [ ]:
from sklearn.model_selection import train_test_split

#Split the dataset into 70% training and 30% temporary data(could change)
train_df, temp_df = train_test_split(
    audio_df,
    test_size=0.30,
    random_state=42,
    stratify=audio_df["species"]
)

#Split the remaining 30% equally into validation and testing data
# 15% validation, 15% testing
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["species"]
)

#check the number of recordings in each dataset
print(f"training recordings: {len(train_df)}")
print(f"validation recordings: {len(val_df)}")
print(f"testing recordings: {len(test_df)}")

#check the distribution of bird species within each category
print("\ntraining:")
print(train_df["species"].value_counts())

print("\nvalidation:")
print(val_df["species"].value_counts())

print("\ntesting:")
print(test_df["species"].value_counts())

Function for extracting MFCC features and returns them

Current feature count: 15

In [ ]:
import numpy as np
import pandas as pd
import librosa

from tqdm.notebook import tqdm


# Extract audio features from one recording
def extract_mfcc(filepath):

    # Load audio
    y, sr = librosa.load(
        filepath,
        sr=22050,
        mono=True
    )

    # Skip recordings that are too short
    if len(y) < 2048:
        raise ValueError("Audio recording is too short")

    # ----------------------------------
    # 1. MFCC FEATURES
    # ----------------------------------

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=20
    )

    # Calculate first-order delta
    mfcc_delta = librosa.feature.delta(
        mfcc,
        order=1
    )

    # Calculate second-order delta
    mfcc_delta2 = librosa.feature.delta(
        mfcc,
        order=2
    )

    # MFCC statistics
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1) 

    # Delta-delta statistics
    delta2_mean = np.mean(mfcc_delta2, axis=1)
    delta2_std = np.std(mfcc_delta2, axis=1)



    # ----------------------------------
    # 9. COMBINE ALL FEATURES
    # ----------------------------------

    features = np.concatenate([

        # MFCC: 40
        mfcc_mean,
        mfcc_std,

        # MFCC Delta-Delta: 40
        delta2_mean,
        delta2_std

    ])

    return features

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

#Extract MFCC features from a dataset
def process_dataset(dataset):

    features_list = []
    labels = []
    filepaths = []
    failed_files = []

    for row in tqdm(
        dataset.itertuples(index=False),
        total=len(dataset),
        desc="extracting MFCC"
    ):
        try:
            #Extract the 40 MFCC features
            features = extract_mfcc(row.filepath)

            #Check that all features contain valid numbers
            if not np.all(np.isfinite(features)):
                raise ValueError("Invalid MFCC values")

            features_list.append(features)
            labels.append(row.species)
            filepaths.append(row.filepath)

        except Exception as e:
            failed_files.append((row.filepath, str(e)))

    #Create the DataFrame with the extracted features
    feature_columns = (
        # 20 + 20
        [f"mfcc_mean_{i}" for i in range(1, 21)] +
        [f"mfcc_std_{i}" for i in range(1, 21)] +

        # MFCC Delta-Delta (40)
        [f"delta2_mean_{i}" for i in range(1, 21)] +
        [f"delta2_std_{i}" for i in range(1, 21)] 


    )

    mfcc_df = pd.DataFrame(
        features_list,
        columns=feature_columns
    )

    #Add the species label and original filepath
    mfcc_df["species"] = labels
    mfcc_df["filepath"] = filepaths

    return mfcc_df, failed_files


#Extract features separately for each dataset
train_mfcc, train_failed = process_dataset(train_df)
val_mfcc, val_failed = process_dataset(val_df)
test_mfcc, test_failed = process_dataset(test_df)

print("\nMFCC Extraction Summary")
print(f"training: {len(train_mfcc)} recordings, {len(train_failed)} failed")
print(f"validation: {len(val_mfcc)} recordings, {len(val_failed)} failed")
print(f"testing: {len(test_mfcc)} recordings, {len(test_failed)} failed")

#Saave stracted features
#each dataset with its MFCC features and species labels
train_mfcc.to_csv("bird_train_mfcc.csv", index=False)
val_mfcc.to_csv("bird_validation_mfcc.csv", index=False)
test_mfcc.to_csv("bird_test_mfcc.csv", index=False)

print("MFCC datasets saved")

#View the training features
display(train_mfcc.head())

making sure all columns are visible

In [ ]:
print(f"Total columns: {train_mfcc.shape[1]}")
print(f"Numeric feature columns: {train_mfcc.select_dtypes(include="number").shape[1]}")
print("All column names:")
print(train_mfcc.columns.tolist())

Separate features from labels   

NOTE: using too many features can cause overfitting so adjust as needed

In [ ]:
# extract column names (features) 

feature_columns = [col for col in train_mfcc.columns if col.startswith("mfcc_") or 
                   col.startswith("delta") or 
                   col.startswith("delta2") or
                   col.startswith("centroid") or
                   col.startswith("bandwidth") or
                   col.startswith("rolloff") or
                   col.startswith("zcr") or
                   col.startswith("rms") or
                   col.startswith("contrast") or
                   col.startswith("chroma")]


# isolate the features and labels for the 3 datasets
X_train = train_mfcc[feature_columns]
y_train = train_mfcc["species"]

X_val = val_mfcc[feature_columns]
y_val = val_mfcc["species"]

X_test = test_mfcc[feature_columns]
y_test = test_mfcc["species"]

print("Training features shape:", X_train.shape)
print("Validation features shape:", X_val.shape)
print("Test features shape:", X_test.shape)

In [ ]:
print(X_train.columns.tolist()) # 120 columns

In [ ]:
# number of neighbors to consider for KNN
k_values = [1, 3, 5, 7, 9, 11] 

# how much influence each neighbor has on the prediction (uniform = equal, distance = closer neighbors have more influence)
weights_options = ['uniform', 'distance']

# distance calculation methods
distance_metrics = ['euclidean', 'manhattan', 'cosine']


# stores results of each hyperparameter combination
tune_results = []

Test all configs    

AI Usage: 

- Used ChatGPT to understand ways to find the best combination of parameters and understand what they mean
- Fix plotting bugs

In [ ]:
# test every combination
for k in k_values:
    for weight in weights_options:
        for metric in distance_metrics:
            print(f"Training KNN with k={k}, weight={weight}, metric={metric}")

            model = Pipeline([
                ("scaler", StandardScaler()),
                ("knn", KNeighborsClassifier(n_neighbors=k, weights=weight, metric=metric))
            ])
            # fit the model on the training data
            model.fit(X_train, y_train)

            # predict validation data
            predictions = model.predict(X_val)

            # calculate accuracy 
            accuracy = accuracy_score(y_val, predictions)
            print(f"Accuracy: {accuracy}")

            # calculate precision, recall, and f1 score
            # macro = calculates precision for individual classes and averages them
            precision = precision_score(y_val, predictions, average='macro', zero_division=0)

            f1 = f1_score(y_val, predictions, average='macro', zero_division=0)

            # list that stores dictionaries of results
            tune_results.append({
                "K": k,
                "Weight": weight,
                "Distance_Metric": metric,
                "Accuracy": accuracy,
                "Precision": precision,
                "F1_Score": f1
            })

# create a DataFrame from the results       
results_df = pd.DataFrame(tune_results)

# sort by F1 score (combines precision and recall to find the best model)
results_df = results_df.sort_values(by="F1_Score", ascending=False)

# display top 10 models
display(results_df.head(10))

Visualize results

In [ ]:
results_df

In [ ]:
# store values of best hyperparameter values in variables for later use

best_model = results_df.iloc[0]
best_k = int(best_model.K)
best_weight = best_model.Weight
best_metric = best_model.Distance_Metric


In [ ]:
best_k, best_weight, best_metric

In [ ]:
# Visual to show the effect of hyperparameters on F1 score

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=results_df,
    x="K",
    y="F1_Score",
    hue="Weight",
    style="Distance_Metric",
    markers=True,
    dashes=False
)

plt.title("KNN Hyperparameter Tuning")
plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Validation Macro F1 Score")

plt.xticks(k_values)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Train the KNN model

In [ ]:
# combine feature and label data for training and validation sets to create a final training set
X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

print("Final training set shape:", X_train_final.shape, y_train_final.shape)

In [ ]:
final_knn = Pipeline([
    ("scaler", StandardScaler()),

    ("knn", KNeighborsClassifier(n_neighbors= best_k, weights=best_weight, metric=best_metric))
])

# train the final model on the combined training and validation data
final_knn.fit(X_train_final, y_train_final)

Saves the scaler and classifier so you don't hve to fit them again when making predictions

In [ ]:
# Save the complete pipeline
joblib.dump(
    {
        "model": final_knn,
        "feature_columns": feature_columns,
        "species": sorted(y_train.unique())
    },
    "best_knn_model.joblib"
)

print("Model saved")

Evaluating the Model

In [ ]:
# making predictions on testing data

y_pred = final_knn.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print("RESULTS")


print(f"Accuracy:  {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")